In [ ]:
from google.colab import userdata  # keys are stored in Colab Secrets
# ==========================================
#  COMPLETE SYSTEM SETUP
# ==========================================

print(" Step 1: Installing libraries...")
!pip install -q sentence-transformers langchain-chroma langchain-community google-genai pypdf python-telegram-bot pyngrok nest-asyncio uvicorn fastapi

print("\n Step 2: Importing libraries...")
import base64  # <--- AJOUTE CETTE LIGNE ICI
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from google import genai
from google.colab import drive
import os

print("\n Step 3: Mounting Google Drive...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("\n Step 4: Setting up embedding class...")
class SentenceTransformerEmbeddings:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

print("\n Step 5: Loading database...")
db_path = "/content/drive/MyDrive/Ramadan_RAG_Data/ramadan_db_v0"
embeddings = SentenceTransformerEmbeddings()
db = Chroma(persist_directory=db_path, embedding_function=embeddings)

print("\n⏳ Step 6: Connecting to Gemini AI...")
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

print("\n" + "=" * 60)
print(" SYSTEM READY!")
print(f" Database loaded: {db._collection.count()} chunks")
print("=" * 60)

In [ ]:
print(" Installing API components...")
!pip install -q fastapi pyngrok nest-asyncio uvicorn
print(" Done!")

In [ ]:
# ==========================================
#  CELL 2: CREATE THE API
# ==========================================
import base64
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional  # <--- La correction est ici
from google.genai import types
# --- API Setup ---
app = FastAPI(
    title="Ramadan Diabetes Expert API",
    description="Multilingual diabetes fasting consultation API",
    version="1.0"
)

# --- Request/Response Models ---
class Question(BaseModel):
    text: str
    image: Optional[str] = None # L'image doit être optionnelle

class Answer(BaseModel):
    response: str
    status: str

# --- RAG FUNCTION ---
# --- RAG FUNCTION ---
def doctor_smart_response(user_input: str, image_b64: Optional[str] = None):
    try:
        # 1. Recherche RAG
        print(f"🔍 USER INPUT: '{user_input}'")
        docs = db.similarity_search(user_input, k=4)
        context = "\n\n".join([d.page_content for d in docs])

        # 2. Le Prompt Multilingue
        prompt = f"""Tu es un expert médical spécialisé dans le Diabète et le jeûne du Ramadan.

RÈGLE ABSOLUE LINGUISTIQUE :
Détecte la langue ou le dialecte utilisé dans la "QUESTION DU PATIENT" (ex: Français, Arabe, Darija).
Tu dois IMPÉRATIVEMENT formuler TOUTE ta réponse dans cette EXACTE MÊME LANGUE.

STRUCTURE OBLIGATOIRE :
🔴 STATUT : (Urgent / Prudence / Sûr)
✅ ACTION : (Consigne claire et courte)
📑 POURQUOI : (Une phrase médicale basée sur le contexte)
💡 CONSEIL : (Une astuce rapide)
⚠️ AVERTISSEMENT : Je suis une IA, veuillez valider avec votre médecin.

CONTEXTE MÉDICAL : {context}
QUESTION DU PATIENT : {user_input}"""

        # 3. Préparation des données (Texte + Image potentielle)
        contents = [prompt]

        if image_b64:
            image_bytes = base64.b64decode(image_b64)
            image_part = types.Part.from_bytes(
                data=image_bytes,
                mime_type='image/jpeg',
            )
            contents.append(image_part)

        # 4. APPEL À L'IA (La ligne qui manquait !)
        response = client.models.generate_content(
            model='gemini-2.0-flash', # Fonctionne parfaitement pour le multilingue et la vision
            contents=contents
        )

        print(f"✅ RESPONSE: {response.text[:100]}...")
        return response.text

    except Exception as e:
        print(f"❌ Erreur: {e}")
        return f"Smahli, mouchkil: {str(e)}"

# --- API ENDPOINTS ---
@app.get("/")
async def root():
    """API home page"""
    return {
        "message": "Ramadan Diabetes Expert API",
        "status": "online",
        "version": "1.0",
        "endpoints": {
            "GET /": "This page",
            "GET /health": "Check API health",
            "POST /ask": "Ask a medical question"
        },
        "example": {
            "endpoint": "/ask",
            "method": "POST",
            "body": {"text": "3endi 60 fssekar"}
        }
    }

@app.get("/health")
async def health():
    """Health check endpoint"""
    try:
        count = db._collection.count()
        return {
            "status": "healthy",
            "database_chunks": count,
            "model": "gemini-2.5-flash"
        }
    except Exception as e:
        return {
            "status": "unhealthy",
            "error": str(e)
        }

@app.post("/ask", response_model=Answer)
async def ask_question(question: Question):
    try:
        # On passe maintenant le texte ET l'image (si elle existe)
        answer = doctor_smart_response(question.text, question.image)

        return {
            "response": answer,
            "status": "success"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("=" * 60)
print(" FastAPI app created successfully!")
print(" RAG function loaded!")
print("=" * 60)

In [ ]:
from google.colab import userdata  # keys are stored in Colab Secrets
# ==========================================
#  CELL 3: LAUNCH API WITH NGROK
# ==========================================

from pyngrok import ngrok
import nest_asyncio
import uvicorn

NGROK_TOKEN = userdata.get('NGROK_TOKEN')

print(" Starting Ngrok tunnel...")

# Kill any old tunnels
ngrok.kill()

# Set token
ngrok.set_auth_token(NGROK_TOKEN)

# Create tunnel on port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "=" * 70)
print(" API IS LIVE!")
print("=" * 70)
print(f" Public URL: {public_url}")
print(f"\n Endpoints:")
print(f"   • Home:        {public_url}/")
print(f"   • Health:      {public_url}/health")
print(f"   • Ask:         {public_url}/ask")
print(f"   • Docs:        {public_url}/docs")
print("\n Test with curl:")
print(f'curl -X POST "{public_url}/ask" -H "Content-Type: application/json" -d \'{{"text":"salam"}}\'')
print(f"\n For Telegram Bot: {public_url}/ask")
print("=" * 70)
print("\n  Keep this cell running to keep API alive!")
print("=" * 70)

# Apply nest_asyncio for Colab compatibility
nest_asyncio.apply()

# Start server
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run server
await server.serve()